# Atividade Prática MongoDB Atlas + Python — Dengue no Brasil (2022–2025)
**Disciplina:** Banco de Dados Não Relacional — Prof. Vagner Macedo

## Fonte dos dados
- **Ministério da Saúde — SINAN (Sistema de Informação de Agravos de Notificação), Dengue 2022–2025**, publicado no OpenDataSUS: https://opendatasus.saude.gov.br/dataset/arboviroses-dengue
- Nomes dos municípios: API de localidades do IBGE (https://servicodados.ibge.gov.br/api/v1/localidades/municipios)



In [7]:
!pip install -q pymongo folium plotly pandas requests

In [18]:

DB_NAME, COL_NAME = "dengue_br", "casos"

UF = {
    11: ("RO", "Norte"), 12: ("AC", "Norte"), 13: ("AM", "Norte"), 14: ("RR", "Norte"),
    15: ("PA", "Norte"), 16: ("AP", "Norte"), 17: ("TO", "Norte"),
    21: ("MA", "Nordeste"), 22: ("PI", "Nordeste"), 23: ("CE", "Nordeste"),
    24: ("RN", "Nordeste"), 25: ("PB", "Nordeste"), 26: ("PE", "Nordeste"),
    27: ("AL", "Nordeste"), 28: ("SE", "Nordeste"), 29: ("BA", "Nordeste"),
    31: ("MG", "Sudeste"), 32: ("ES", "Sudeste"), 33: ("RJ", "Sudeste"), 35: ("SP", "Sudeste"),
    41: ("PR", "Sul"), 42: ("SC", "Sul"), 43: ("RS", "Sul"),
    50: ("MS", "Centro-Oeste"), 51: ("MT", "Centro-Oeste"), 52: ("GO", "Centro-Oeste"),
    53: ("DF", "Centro-Oeste"),
}

CENTROS = {
    "Norte": (-3.9, -61.4), "Nordeste": (-9.6, -40.5), "Centro-Oeste": (-15.6, -54.3),
    "Sudeste": (-20.5, -45.0), "Sul": (-26.3, -51.5),
}

CIDADES_SUL = {
    410690: ("Curitiba", "PR", -25.4284, -49.2733),
    411370: ("Londrina", "PR", -23.3045, -51.1696),
    411520: ("Maringá", "PR", -23.4205, -51.9331),
    420540: ("Florianópolis", "SC", -27.5954, -48.5480),
    420910: ("Joinville", "SC", -26.3045, -48.8487),
    420240: ("Blumenau", "SC", -26.9194, -49.0661),
    431490: ("Porto Alegre", "RS", -30.0346, -51.2177),
    430510: ("Caxias do Sul", "RS", -29.1678, -51.1794),
    431440: ("Pelotas", "RS", -31.7654, -52.3376),
    430460: ("Canoas", "RS", -29.9178, -51.1839),
}


## 1. Coleta e agregação dos dados (SINAN → ano × município)
Os arquivos têm milhões de linhas; por isso são lidos em *chunks*, apenas com as colunas necessárias, e agregados na hora.
`ID_MUNICIP` é o código IBGE de 6 dígitos do município de notificação; seus 2 primeiros dígitos identificam a UF.

In [19]:
import io, json, zipfile, requests
import pandas as pd

ANOS = [2022, 2023, 2024, 2025]
URL = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR{yy}.csv.zip"
COLS = ["ID_MUNICIP", "CLASSI_FIN"]

def agregar_ano(ano, chunksize=500_000):
    """Baixa o zip do ano e devolve Series: casos prováveis por município (código de 6 dígitos)."""
    destino = f"DENGBR{ano % 100}.csv.zip"
    with requests.get(URL.format(yy=ano % 100), stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for parte in r.iter_content(1024 * 1024):
                f.write(parte)
    total = pd.Series(dtype="int64")

    for enc in ("utf-8", "latin-1"):
        try:
            total = pd.Series(dtype="int64")
            for ch in pd.read_csv(destino, compression="zip", usecols=COLS, chunksize=chunksize,
                                  dtype=str, encoding=enc, low_memory=False):
                ch = ch.dropna(subset=["ID_MUNICIP"])
                ch = ch[ch["CLASSI_FIN"].fillna("") != "5"]
                total = total.add(ch["ID_MUNICIP"].astype(int).value_counts(), fill_value=0)
            break
        except UnicodeDecodeError:
            continue
    return total.astype(int)

agregados = {}
for ano in ANOS:
    agregados[ano] = agregar_ano(ano)
    print(ano, "→", f"{agregados[ano].sum():,} casos prováveis em {len(agregados[ano])} municípios")

2022 → 1,405,095 casos prováveis em 4678 municípios
2023 → 1,645,956 casos prováveis em 4615 municípios
2024 → 6,564,924 casos prováveis em 5291 municípios
2025 → 1,643,215 casos prováveis em 4822 municípios


In [10]:

try:
    mun = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/municipios", timeout=60).json()
    NOMES = {int(str(m["id"])[:6]): m["nome"] for m in mun}
except Exception as e:
    print("Não foi possível obter nomes no IBGE:", e)
    NOMES = {}
NOMES.update({k: v[0] for k, v in CIDADES_SUL.items()})

docs = []
for ano, serie in agregados.items():
    for cod, casos in serie.items():
        uf_cod = cod // 10000
        if uf_cod not in UF:
            continue
        sigla, regiao = UF[uf_cod]
        docs.append({"ano": ano, "regiao": regiao, "uf": sigla, "id_municipio": int(cod),
                     "municipio": NOMES.get(int(cod)), "casos": int(casos)})

with open("dengue_2022_2025.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False)
print(len(docs), "documentos gravados em dengue_2022_2025.json")
pd.DataFrame(docs).head()

19406 documentos gravados em dengue_2022_2025.json


,ano,regiao,uf,id_municipio,municipio,casos
0,2022,Norte,RO,110001,Alta Floresta D'Oeste,374
1,2022,Norte,RO,110002,Ariquemes,1116
2,2022,Norte,RO,110003,Cabixi,26
3,2022,Norte,RO,110004,Cacoal,2193
4,2022,Norte,RO,110005,Cerejeiras,26


## 2. Importação no MongoDB Atlas
Cada documento = casos prováveis de um município em um ano. Cole sua *connection string* quando solicitado
(Atlas → *Connect* → *Drivers*; libere o IP `0.0.0.0/0` em *Network Access* para usar no Colab).

In [11]:
from getpass import getpass
from urllib.parse import quote_plus
from pymongo import MongoClient

usuario = input("Usuário do banco (Database Access): ").strip()
senha = getpass("Senha do usuário: ")
host = input("Host do cluster (ex.: cluster0.abcde.mongodb.net): ").strip()

MONGO_URI = f"mongodb+srv://{quote_plus(usuario)}:{quote_plus(senha)}@{host}/?retryWrites=true&w=majority"
client = MongoClient(MONGO_URI)
client.admin.command("ping")
col = client[DB_NAME][COL_NAME]

with open("dengue_2022_2025.json", encoding="utf-8") as f:
    docs = json.load(f)

col.drop()
for i in range(0, len(docs), 10_000):
    col.insert_many(docs[i:i + 10_000])
col.create_index([("regiao", 1), ("uf", 1)])
col.create_index("id_municipio")
print("Importados:", col.count_documents({}), "documentos")

Usuário do banco (Database Access): vitoriamillena453_db_user
Senha do usuário: ··········
Host do cluster (ex.: cluster0.abcde.mongodb.net): millena.omtd0kd.mongodb.net
Importados: 19406 documentos


## 3. Soma de casos por região
*Aggregation pipeline* do MongoDB: `$group` por região somando `casos`.

In [12]:
pipe_regiao = [{"$group": {"_id": "$regiao", "total": {"$sum": "$casos"}}},
               {"$sort": {"total": -1}}]
total_regiao = pd.DataFrame(col.aggregate(pipe_regiao)).rename(columns={"_id": "regiao"})
total_regiao

,regiao,total
0,Sudeste,6793680
1,Sul,2126231
2,Centro-Oeste,1366702
3,Nordeste,792858
4,Norte,179719


## 4. Mapa folium — ponto central de cada região com a soma de casos
O raio do círculo é proporcional à raiz quadrada do total (para a área refletir o valor).

In [20]:
import folium

def mapa_regioes(df):
    m = folium.Map(location=[-14.5, -52], zoom_start=4, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}", attr="Esri")
    maximo = df["total"].max()
    for _, r in df.iterrows():
        lat, lon = CENTROS[r["regiao"]]
        folium.CircleMarker(
            [lat, lon], radius=8 + 32 * (r["total"] / maximo) ** 0.5, color="#b30000",
            fill=True, fill_opacity=0.45,
            tooltip=f"{r['regiao']}: {int(r['total']):,} casos".replace(",", "."),
        ).add_to(m)
        folium.Marker([lat, lon], icon=folium.DivIcon(html=(
            f"<div style='font:700 12px sans-serif;text-align:center;width:120px;margin-left:-60px;"
            f"margin-top:-8px'>{r['regiao']}<br>{int(r['total']):,}</div>".replace(",", ".")))).add_to(m)
    return m

mapa_regioes(total_regiao)

## 5. Média de casos por região
Primeiro soma-se os casos de **cada estado**; depois calcula-se, por região, a **média entre os estados** (dois `$group` encadeados).

In [21]:
import plotly.express as px

pipe_media = [
    {"$group": {"_id": {"regiao": "$regiao", "uf": "$uf"}, "total_estado": {"$sum": "$casos"}}},
    {"$group": {"_id": "$_id.regiao", "media_por_estado": {"$avg": "$total_estado"},
                "n_estados": {"$sum": 1}}},
    {"$sort": {"media_por_estado": -1}},
]
media_regiao = pd.DataFrame(col.aggregate(pipe_media)).rename(columns={"_id": "regiao"})
display(media_regiao)

fig = px.bar(media_regiao, x="regiao", y="media_por_estado", text_auto=",.0f",
             labels={"regiao": "Região", "media_por_estado": "Média de casos por estado"},
             title="Média de casos de dengue por estado, por região (2022–2025)")
fig.show()

,regiao,media_por_estado,n_estados
0,Sudeste,1.698420e+06,4
1,Sul,7.087437e+05,3
2,Centro-Oeste,3.416755e+05,4
3,Nordeste,8.809533e+04,9
4,Norte,2.567414e+04,7


## 6. Região Sul — ocorrências em 10 grandes cidades
Filtro por `id_municipio` (código IBGE de 6 dígitos) e soma dos casos de 2022 a 2025.

In [22]:
pipe_sul = [
    {"$match": {"regiao": "Sul", "id_municipio": {"$in": list(CIDADES_SUL)}}},
    {"$group": {"_id": "$id_municipio", "casos": {"$sum": "$casos"}}},
    {"$sort": {"casos": -1}},
]
cidades = pd.DataFrame(col.aggregate(pipe_sul)).rename(columns={"_id": "id_municipio"})
cidades["cidade"] = cidades["id_municipio"].map(lambda c: f"{CIDADES_SUL[c][0]}/{CIDADES_SUL[c][1]}")
display(cidades[["cidade", "casos"]])

def mapa_cidades(df):
    m = folium.Map(location=[-27.8, -51.0], zoom_start=6, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}", attr="Esri")
    maximo = max(df["casos"].max(), 1)
    for _, r in df.iterrows():
        nome, uf, lat, lon = CIDADES_SUL[r["id_municipio"]]
        folium.CircleMarker([lat, lon], radius=6 + 28 * (r["casos"] / maximo) ** 0.5,
                            color="#b30000", fill=True, fill_opacity=0.5,
                            tooltip=f"{nome}/{uf}: {int(r['casos']):,} casos".replace(",", ".")).add_to(m)
    return m

mapa_cidades(cidades)

,cidade,casos
0,Joinville/SC,151961
1,Londrina/PR,108517
2,Porto Alegre/RS,100174
3,Blumenau/SC,73840
4,Florianópolis/SC,70273
5,Maringá/PR,44479
6,Curitiba/PR,22483
7,Canoas/RS,10448
8,Pelotas/RS,716
9,Caxias do Sul/RS,451
